In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle

import structlog
import logging
structlog.configure(
    wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING),
)

import sys
sys.path.append('../../../')

import pickle
import d3rlpy

from d3rlpy.algos import BCConfig, IQLConfig, CQLConfig
from src.difsched.agents.gym_env import HybridEnv
from src.difsched.config import getExpConfig, visualizeExpConfig
from src.difsched.env.Hybrid import createEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
c:\Users\Ye\miniconda3\envs\traffic_predictor_3_9\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def evaluate_with_stats(algo, env, n_episodes=5):
    """Evaluate algorithm and return mean and std of average rewards per episode."""
    from src.difsched.agents.dr3rlpy.evaluation import reset_compat, step_compat
    import gymnasium as gym
    
    returns = []
    for _ in range(n_episodes):
        obs = reset_compat(env)
        done = False
        ep_ret = 0.0
        step_count = 0
        while not done:
            act = algo.predict(obs[None, ...])[0]
            if isinstance(env.action_space, gym.spaces.Discrete):
                act = int(act)
            obs, r, done, *_ = step_compat(env, act)
            ep_ret += r
            step_count += 1
        avg_reward = ep_ret / step_count if step_count > 0 else 0.0
        returns.append(avg_reward)
    
    mean_reward = np.mean(returns)
    std_reward = np.std(returns)
    return mean_reward, std_reward

In [3]:
model_dir = "../../../data/results/d3rlpy"
trafficDatasetFolder = f'../../../data/processed/traffic'


In [4]:
results = []

for expConfigIdx in range(7, 8):
    expParams = getExpConfig(expConfigIdx)
    visualizeExpConfig(expParams)
    simEnv = createEnv(expParams, trafficDatasetFolder)
    simEnv.selectMode(mode="test", type="data")

    bc_path = os.path.join(model_dir, f"bc_model_exp{expConfigIdx}.d3")
    iql_path = os.path.join(model_dir, f"iql_model_exp{expConfigIdx}.d3")
    cql_path = os.path.join(model_dir, f"cql_model_exp{expConfigIdx}.d3")

    print(f"Loading models for experiment config {expConfigIdx}...")
    try:
        from d3rlpy.algos import BC, IQL, CQL
        bc = BC.from_json(bc_path, device="cuda:0")
        print(f"BC model loaded from {bc_path}")
        
        iql = IQL.from_json(iql_path, device="cuda:0")
        print(f"IQL model loaded from {iql_path}")
        
        cql = CQL.from_json(cql_path, device="cuda:0")
        print(f"CQL model loaded from {cql_path}")
    except:
        print("from_json not available, using load_learnable...")
        bc = d3rlpy.load_learnable(bc_path, device="cuda:0")
        print(f"BC model loaded from {bc_path}")
        
        iql = d3rlpy.load_learnable(iql_path, device="cuda:0")
        print(f"IQL model loaded from {iql_path}")
        
        cql = d3rlpy.load_learnable(cql_path, device="cuda:0")
        print(f"CQL model loaded from {cql_path}")

    print("All models loaded successfully!")

    max_episode_steps = 250
    evaluate_ep = 10

    env = HybridEnv(expParams, simEnv, obvMode="predicted", max_episode_steps=max_episode_steps)
    reward_mean_bc, reward_std_bc = evaluate_with_stats(bc, env, n_episodes=evaluate_ep)
    packet_loss_bc = 1 - reward_mean_bc
    print(f"BC avg packet loss rate: {1-reward_mean_bc:.4f} ± {reward_std_bc:.4f}")

    env = HybridEnv(expParams, simEnv, obvMode="predicted", max_episode_steps=max_episode_steps)
    reward_mean_iql, reward_std_iql = evaluate_with_stats(iql, env, n_episodes=evaluate_ep)
    packet_loss_iql = 1 - reward_mean_iql
    print(f"IQL avg packet loss rate: {1-reward_mean_iql:.4f} ± {reward_std_iql:.4f}")

    env = HybridEnv(expParams, simEnv, obvMode="predicted", max_episode_steps=max_episode_steps)
    reward_mean_cql, reward_std_cql = evaluate_with_stats(cql, env, n_episodes=evaluate_ep)
    packet_loss_cql = 1 - reward_mean_cql
    print(f"CQL avg packet loss rate: {1-reward_mean_cql:.4f} ± {reward_std_cql:.4f}")
    
    results.append({
        'Exp_ID': expConfigIdx,
        'N_user': expParams['N_user'],
        'dataflow': expParams['dataflow'],
        'LEN_window': expParams['LEN_window'],
        'N_aggregation': expParams['N_aggregation'],
        'r_bar': expParams['r_bar'],
        'B': expParams['B'],
        'BC_Reward': reward_mean_bc,
        'BC_Reward_Std': reward_std_bc,
        'BC_PacketLoss': packet_loss_bc,
        'IQL_Reward': reward_mean_iql,
        'IQL_Reward_Std': reward_std_iql,
        'IQL_PacketLoss': packet_loss_iql,
        'CQL_Reward': reward_mean_cql,
        'CQL_Reward_Std': reward_std_cql,
        'CQL_PacketLoss': packet_loss_cql
    })

df_results = pd.DataFrame(results)

EnvType: HYBRID
N_user: 32
LEN_window: 100
N_aggregation: 4
dataflow: haptic_1ms_100
randomSeed: 999
r_bar: 4
B: 180
sigma_list: [0.7, 0.75, 0.8, 0.85, 0.9]
offline_dataset_idxs: [21, 22, 23]
Loading models for experiment config 7...
from_json not available, using load_learnable...


c:\Users\Ye\miniconda3\envs\traffic_predictor_3_9\lib\site-packages\d3rlpy\torch_utility.py:431: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  chkpt = torch.load(f, map_loca

BC model loaded from ../../../data/results/d3rlpy\bc_model_exp7.d3
IQL model loaded from ../../../data/results/d3rlpy\iql_model_exp7.d3
CQL model loaded from ../../../data/results/d3rlpy\cql_model_exp7.d3
All models loaded successfully!
BC avg packet loss rate: 0.0225 ± 0.0012
IQL avg packet loss rate: 0.0172 ± 0.0005
CQL avg packet loss rate: 0.0197 ± 0.0008


In [5]:
print("\n" + "="*100)
print("EVALUATION RESULTS SUMMARY")
print("="*100 + "\n")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

summary_df = pd.DataFrame({
    'ID': df_results['Exp_ID'],
    'dataflow': df_results['dataflow'],
    'N_user': df_results['N_user'],
    'IQL_PacketLoss': df_results.apply(lambda row: f"{row['IQL_PacketLoss']:.4f}±{row['IQL_Reward_Std']:.4f}", axis=1),
    'CQL_PacketLoss': df_results.apply(lambda row: f"{row['CQL_PacketLoss']:.4f}±{row['CQL_Reward_Std']:.4f}", axis=1),
    'BC_PacketLoss': df_results.apply(lambda row: f"{row['BC_PacketLoss']:.4f}±{row['BC_Reward_Std']:.4f}", axis=1)
})

summary_df = summary_df.sort_values(by=['dataflow', 'N_user'])

print(summary_df.to_string(index=False))



EVALUATION RESULTS SUMMARY

 ID       dataflow  N_user IQL_PacketLoss CQL_PacketLoss BC_PacketLoss
  7 haptic_1ms_100      32  0.0172±0.0005  0.0197±0.0008 0.0225±0.0012
